# GdeltForge: getting started

GdeltForge turns the raw [GDELT](https://www.gdeltproject.org/) archive (news
events, mentions, and the Global Knowledge Graph) into clean, reproducible
Parquet files you can actually work with, without hand-rolling a scraper or
learning GDELT's raw CSV formats.

This notebook runs the real pipeline end to end against a small, live slice
of GDELT data: install, scrape, convert, filter, sample, and (optionally)
cross-reference against the Global Knowledge Graph. It works the same way
whether you're running it here in the repo or as a standalone Colab notebook.

**What you need:** Python 3.10+, internet access, and about a minute for the
core pipeline (the optional GKG section downloads more data and takes a few
minutes longer). Every command below is the real `gdeltforge` CLI: nothing
in this notebook is simulated.

For the full reference (every command, every config option, every sampling
mode) see the [documentation site](https://vinicius-teixeirac.github.io/GdeltForge/)
and the [GitHub repo](https://github.com/Vinicius-Teixeirac/GdeltForge).

## Setup

In [ ]:
%pip install -q gdeltforge

In [ ]:
!gdeltforge --version

`--help` lists every stage of the pipeline. We'll walk through each one below.

In [ ]:
!gdeltforge --help

## The pipeline

GdeltForge is five commands in sequence:

1. **scrape** downloads raw GDELT files for a date range.
2. **convert** turns them into Parquet.
3. **filter** applies any row/column rules you've configured (a no-op by default).
4. **sample** draws a manageable, reproducible sample from the filtered data.
5. **crossref** (optional) joins a sampled Events file against GKG or Mentions for context.

The first command you run creates `config/settings.yaml` with conservative
defaults (no filtering, no column pruning) so a first run never silently
drops data. You'll see a one-time warning about this below; that's expected.

### Pick a date

GDELT publishes continuously, so a request for "the last few hours" would
give a different answer every time you ran this notebook. We'll pull a date
a few days in the past instead: it's guaranteed to have finished publishing,
and everyone who runs this notebook gets the same day.

In [ ]:
from datetime import date, timedelta

START_DATE = (date.today() - timedelta(days=5)).isoformat()
print(f"Using GDELT data from {START_DATE}")

## Quick path: Events only

Events is GDELT's core daily dataset, one file per day, so this is the
fastest way to see the whole pipeline work.

### 1. Scrape

In [ ]:
!gdeltforge scrape --dataset events --start-date {START_DATE}

### 2. Convert

Turns the downloaded ZIP into Parquet. `--quiet` suppresses the per-file
log lines; drop it if you want to see them.

In [ ]:
!gdeltforge convert --dataset events --quiet

### 3. Filter

Applies whatever row/column rules are configured in `config/settings.yaml`.
With the bundled defaults this is a pass-through (100% retention); it's
still a required stage since `sample` reads from filtered output.

In [ ]:
!gdeltforge filter --dataset events

### 4. Sample

`--mode indexed` draws a uniform random sample across every row in the
filtered dataset. `--seed` makes it reproducible.

In [ ]:
!gdeltforge sample --dataset events --mode indexed -n 1000 --seed 123 --out sample.parquet

## Explore your sample

GdeltForge outputs plain Parquet, so any tool that reads Parquet works.
We'll use [polars](https://pola.rs/) here since it's already installed as
one of GdeltForge's own dependencies, no extra install needed.

In [ ]:
import polars as pl

events = pl.read_parquet("sample.parquet")
print(f"{events.height} rows, {events.width} columns")
events.head()

In [ ]:
events["GlobalEventID"].n_unique()

### Structuring a single event

Event data packs a lot into one row, often grouped implicitly by column
prefix (`Actor1`, `ActionGeo`, ...). Here's one random row laid out by
category, not as a flat table.

In [ ]:
random_event = events.sample(1).row(0, named=True)

print("### Event")
print(f"Global Event ID: {random_event['GlobalEventID']}")
print(f"Event code: {random_event['EventCode']} (root: {random_event['EventRootCode']})")
print(f"Goldstein scale: {random_event['GoldsteinScale']}")
n_mentions = random_event['NumMentions']
n_sources = random_event['NumSources']
n_articles = random_event['NumArticles']
print(f"Mentions: {n_mentions} (sources: {n_sources}, articles: {n_articles})")
print(f"Average tone: {random_event['AvgTone']}")

print("\n### Actor 1 (who)")
print(f"Name: {random_event['Actor1Name']}")
print(f"Country: {random_event['Actor1CountryCode']}")

print("\n### Actor 2 (whom)")
print(f"Name: {random_event['Actor2Name']}")
print(f"Country: {random_event['Actor2CountryCode']}")

print("\n### Location")
print(f"Name: {random_event['ActionGeo_FullName']}")
print(f"Coordinates: ({random_event['ActionGeo_Lat']}, {random_event['ActionGeo_Long']})")

print("\n### Date")
print(f"Day: {random_event['Day']}")

print(f"\nSource URL: {random_event['SOURCEURL']}")

## What people actually use GDELT for

A few common applications, each built on Events and GKG data like the kind
you're working with here:

- **Conflict and unrest early warning**: tracking protest and violence-coded
  events by country over time to flag emerging situations.
- **Media narrative and bias tracking**: comparing how outlets in different
  countries or languages cover the same event.
- **Financial and geopolitical risk signals**: event-driven sentiment feeding
  trading models or corporate risk scoring.
- **Humanitarian and disaster response**: spotting spikes in crisis-related
  coverage to direct attention or aid.
- **Academic research**: large-scale, quantitative political science and
  international relations studies.

As a small taste of the first one, let's do some "event sensing": use the
full day of Events data, not the 1000-row sample, to see which countries
had the most protest activity that day.

In [ ]:
# EventRootCode 14 is CAMEO's "Protest" category.
full_day = pl.read_parquet("data/events/filtered/*.parquet")

protests_by_country = (
    full_day
    .filter(pl.col("EventRootCode") == "14")
    .filter(pl.col("ActionGeo_CountryCode").is_not_null())
    .group_by("ActionGeo_CountryCode")
    .agg(
        pl.len().alias("protest_events"),
        pl.col("AvgTone").mean().round(2).alias("avg_tone"),
    )
    .sort("protest_events", descending=True)
)

print(f"Protest-coded events on {START_DATE}, by country:")
protests_by_country.head(10)

This is exactly the kind of signal that feeds real GDELT-based unrest
dashboards: a sudden jump in one country's protest count from one day to
the next is the "sensing" part. Doing that properly means pulling several
days of Events (add `--start-date`/`--end-date` to the scrape command
above) and comparing counts over time, left as an exercise for you.

## Going further: enrich with GKG context

Events tells you *what* happened. The [Global Knowledge Graph](https://blog.gdeltproject.org/gdelt-global-knowledge-graph-gkg/)
(GKG) tells you *how it was covered*: themes, people, organizations, and
tone extracted from every article that mentions an event.

`crossref` joins your Events sample against GKG through Mentions (the
table linking events to the articles that reported them). This section
downloads a full day of 15-minute GKG and Mentions files (a few hundred
files each), so it takes a few minutes, not seconds. Skip it if you
just wanted the quick path above.

### 5. Scrape, convert, and filter GKG 2.1 and Mentions

Same three stages as before, just for two more datasets.

In [ ]:
!gdeltforge scrape --dataset gkg-v2 --start-date {START_DATE}
!gdeltforge scrape --dataset mentions --start-date {START_DATE}

In [ ]:
!gdeltforge convert --dataset gkg-v2 --quiet
!gdeltforge convert --dataset mentions --quiet

In [ ]:
!gdeltforge filter --dataset gkg-v2
!gdeltforge filter --dataset mentions

### 6. Cross-reference

Joins the Events sample from earlier against GKG 2.1 (via Mentions). One
event can be covered by several articles, so the output has more rows
than the input sample.

In [ ]:
!gdeltforge crossref \
    --events sample.parquet \
    --gkg-version v2 \
    --out sample_with_gkg.parquet

### Explore the enriched sample

The joined output carries every original Events column plus GKG's themes,
people, organizations, and content-analysis fields for the covering article.

In [ ]:
enriched = pl.read_parquet("sample_with_gkg.parquet")
print(f"{enriched.height} rows, {enriched.width} columns")
enriched.head()

In [ ]:
random_enriched = enriched.sample(1).row(0, named=True)

print("### GKG context for this event")

persons = random_enriched.get("GKG_V1PERSONS")
if persons:
    names = [p.split(",")[0] for p in persons.split(";") if p.strip()]
    print(f"Persons mentioned: {names}")
else:
    print("Persons mentioned: none")

themes = random_enriched.get("GKG_V1THEMES")
if themes:
    theme_list = [t.strip() for t in themes.split(";") if t.strip()]
    print(f"Themes ({len(theme_list)} total): {theme_list[:8]}")
else:
    print("Themes: none")

locations = random_enriched.get("GKG_V2ENHANCEDLOCATIONS")
if locations:
    first = locations.split(";")[0].split("#")
    if len(first) >= 2:
        print(f"First enhanced location: type {first[0]}, name {first[1]}")
else:
    print("Enhanced locations: none")

### GCAM: content analysis measures

`GKG_V2GCAM` packs dozens of `key:value` sentiment and content-analysis
scores into one comma-separated field. Here are the first few, parsed.

In [ ]:
gcam_raw = random_enriched.get("GKG_V2GCAM")
if gcam_raw:
    gcam = dict(item.split(":", 1) for item in gcam_raw.split(",") if ":" in item)
    print(f"Word count (wc): {gcam.get('wc', 'n/a')}")
    print(f"{len(gcam)} GCAM entries total, first 15:")
    for k, v in list(gcam.items())[:15]:
        print(f"  {k}: {v}")
else:
    print("GKG_V2GCAM: none")

## Where to go next

- [CLI reference](https://vinicius-teixeirac.github.io/GdeltForge/cli-reference/): every flag for every command.
- [Configuration guide](https://vinicius-teixeirac.github.io/GdeltForge/configuration/): customizing `config/settings.yaml` (column pruning, compression, worker counts, capacity planning).
- Other sampling modes: `--mode calendar` (day/month/year buckets) and `--mode filtered --stratify` (stratified by a column).
- `gdeltforge codes`: look up valid GDELT country/actor codes for use in filter rules.
- GKG 1.0 (the pre-2015 format, still published daily) and direct Mentions access work the same way, via `--dataset gkg-v1` / `gkg-v1-counts` / `mentions`.
- [CHANGELOG](https://github.com/Vinicius-Teixeirac/GdeltForge/blob/main/CHANGELOG.md) for what's new.

## Clean up (optional)

Everything this notebook downloaded lives under `./data`, plus
`config/settings.yaml`, `sample.parquet`, and `sample_with_gkg.parquet`.
Uncomment and run the next cell to remove it all.

In [ ]:
# import shutil
# from pathlib import Path
#
# shutil.rmtree("data", ignore_errors=True)
# shutil.rmtree("config", ignore_errors=True)
# for f in ("sample.parquet", "sample_with_gkg.parquet"):
#     Path(f).unlink(missing_ok=True)